# Student Notebook 01 — Modeling (Layer 1)

Pick a model, train it, evaluate it. Every teammate runs this
notebook with a different choice in the CONFIG cell below
and shares the headline AUC number.

**Run order:** this is notebook 01. After running it, run
02 → 03 → 04, and only then 05 (the test set).

**What you'll get out:** CV AUC (mean ± std), held-out cal
AUC, a saved model at
``data/processed/student/student_model.joblib`` that the
next notebooks will read.

## CONFIG — edit me

Change any value below and re-run the notebook. The
comments list the supported options.

In [ ]:
CONFIG = {
    # YOUR run_name. Use a unique label so your artifacts don't
    # clobber teammates'. Convention: "<your-name>_<model>".
    # Each run lands in data/processed/student/<run_name>/.
    "run_name": "alice_xgboost",

    # Target. The headline target is roi_gt_2 (was the film
    # net profitable?). roi_gt_1 is the easier 1x ROI binary;
    # log_roi is regression. The other two notebooks assume
    # classification, so for the team comparison stick with
    # roi_gt_2 unless you want to dig into log_roi separately.
    "target": "roi_gt_2",                # "roi_gt_2" | "roi_gt_1" | "log_roi"

    # Model family. Try each one and compare CV AUC.
    "model_family": "xgboost",            # "logistic" | "random_forest" | "xgboost" | "svm_rbf"

    # Feature subset. "all" uses the 127-feature matrix from
    # Phase 3. The others let you isolate one feature group
    # to see how much each one matters.
    "feature_set": "all",                 # "all" | "structural" | "topic" | "embedding" | "network"

    # CV / random seed. Don't change unless you know why.
    "n_splits": 5,
    "random_seed": 42,
}

## Imports and paths

In [ ]:
# --- Paths (works from any working directory) ---
from pathlib import Path

def _find_project_root() -> Path:
    p = Path.cwd().resolve()
    for cand in [p, *p.parents]:
        if (cand / "docs" / "PROJECT_CONTEXT.md").is_file():
            return cand
    raise RuntimeError(f"Could not locate project root from {Path.cwd()!s}")

PROJECT_ROOT = _find_project_root()
DATA = PROJECT_ROOT / "data" / "processed"
STUDENT = DATA / "student" / CONFIG["run_name"]
STUDENT.mkdir(parents=True, exist_ok=True)
print(f"Project root:  {PROJECT_ROOT}")
print(f"Run name:      {CONFIG['run_name']!r}")
print(f"Artifacts go:  {STUDENT.relative_to(PROJECT_ROOT)}")

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, roc_curve
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

try:
    from xgboost import XGBClassifier
    HAS_XGB = True
except ImportError:
    HAS_XGB = False

## Load features and targets

We use the 92-feature matrix that the rigorous pipeline
settled on (``standalone_positive_union_mpnet`` from
Phase 3). Train + cal splits only — the test split is for
notebook 05.

In [ ]:
# Read the prebuilt feature matrix. imdb_id is its index;
# ``split`` is already a column ('train' / 'cal' / 'test').
feat = pd.read_parquet(DATA / "features.parquet").reset_index()
print(f"Total films in the matrix: {len(feat)}")
print(f"Split breakdown: {feat['split'].value_counts().to_dict()}")

# Train + cal pool (the test split is held out for notebook 05).
df = feat[feat["split"].isin(["train", "cal"])].reset_index(drop=True)
print(f"Films available in this notebook: {len(df)}")

# Pick feature columns based on CONFIG['feature_set'].
non_feat = {"imdb_id", "split", "log_roi", "roi_gt_1", "roi_gt_2"}
all_cols = [c for c in df.columns if c not in non_feat]
groups = {
    "all":        all_cols,
    "structural": [c for c in all_cols if c.startswith(("log_", "n_", "dialogue_to_", "release_year", "log_runtime", "genre_"))],
    "topic":      [c for c in all_cols if c.startswith("topic_")],
    "embedding":  [c for c in all_cols if c.startswith("embed_pc_")],
    "network":    [c for c in all_cols if c.startswith("network_")],
}
feat_cols = groups[CONFIG["feature_set"]]
print(f"Feature set: {CONFIG['feature_set']!r} -> {len(feat_cols)} columns")

X = df[feat_cols].fillna(0).values
y = df[CONFIG["target"]].astype(int).values if CONFIG["target"].startswith("roi_gt") else df[CONFIG["target"]].values
print(f"X shape: {X.shape}, y positive rate: {y.mean():.3f}")

## Build the model

Each model family is wrapped in a sklearn ``Pipeline`` with a
``StandardScaler`` so the input distribution is consistent.
SVM also goes through ``probability=True`` so the next
notebook can calibrate it.

In [ ]:
def build_model(family: str, seed: int = 42):
    if family == "logistic":
        clf = LogisticRegression(max_iter=2000, C=1.0, random_state=seed)
    elif family == "random_forest":
        clf = RandomForestClassifier(n_estimators=300, max_depth=None,
                                      min_samples_leaf=2, random_state=seed, n_jobs=-1)
    elif family == "xgboost":
        if not HAS_XGB:
            raise RuntimeError("xgboost not installed. pip install xgboost")
        clf = XGBClassifier(n_estimators=300, max_depth=4, learning_rate=0.05,
                            subsample=0.8, colsample_bytree=0.8,
                            eval_metric="logloss", random_state=seed, n_jobs=-1)
    elif family == "svm_rbf":
        clf = SVC(kernel="rbf", C=1.0, gamma="scale", probability=True, random_state=seed)
    else:
        raise ValueError(f"Unknown model_family: {family!r}")
    return Pipeline([("scaler", StandardScaler(with_mean=False)), ("model", clf)])

model = build_model(CONFIG["model_family"], seed=CONFIG["random_seed"])
print(model)

## 5-fold cross-validation AUC

``cross_val_score`` returns one AUC per fold. We report mean
± std plus a 95% bootstrap CI on the per-fold values. Higher
is better; AUC of 0.5 is chance, AUC of 1.0 is perfect.

In [ ]:
skf = StratifiedKFold(n_splits=CONFIG["n_splits"], shuffle=True, random_state=CONFIG["random_seed"])
cv_aucs = cross_val_score(model, X, y, cv=skf, scoring="roc_auc", n_jobs=-1)
print(f"CV AUC per fold: {[f'{a:.3f}' for a in cv_aucs]}")
print(f"CV AUC mean ± std: {cv_aucs.mean():.3f} ± {cv_aucs.std():.3f}")

# Quick bootstrap CI on the per-fold AUCs.
rng = np.random.default_rng(CONFIG["random_seed"])
boot = np.array([rng.choice(cv_aucs, size=len(cv_aucs), replace=True).mean() for _ in range(2000)])
ci_lo, ci_hi = np.quantile(boot, [0.025, 0.975])
print(f"95% CI: [{ci_lo:.3f}, {ci_hi:.3f}]")

## Held-out evaluation on the cal split

Train on the 1,199 train films, score the 257 cal films.
This is the "what would I see on new data?" estimate that
the next notebooks build on.

In [ ]:
train_mask = (df["split"] == "train").values
cal_mask   = (df["split"] == "cal").values

X_train, y_train = X[train_mask], y[train_mask]
X_cal, y_cal = X[cal_mask], y[cal_mask]

model.fit(X_train, y_train)
cal_proba = model.predict_proba(X_cal)[:, 1]
cal_auc = roc_auc_score(y_cal, cal_proba)
print(f"Cal-set AUC: {cal_auc:.3f}")

## ROC curve on the cal set

In [ ]:
fpr, tpr, _ = roc_curve(y_cal, cal_proba)
plt.figure(figsize=(5, 5))
plt.plot(fpr, tpr, label=f"{CONFIG['model_family']} (AUC = {cal_auc:.3f})", linewidth=2)
plt.plot([0, 1], [0, 1], "--", color="grey", label="chance")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title(f"ROC — {CONFIG['target']} on cal set")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

## Save the model

Saves to ``data/processed/student/student_model.joblib``.
Notebooks 02-05 will load this file. If you re-run this
notebook with a different model, the next notebooks
automatically pick it up.

In [ ]:
bundle = {
    "config": CONFIG,
    "feature_columns": feat_cols,
    "model": model,
    "cv_auc_mean": float(cv_aucs.mean()),
    "cv_auc_std": float(cv_aucs.std()),
    "cal_auc": float(cal_auc),
    "n_train": int(train_mask.sum()),
    "n_cal": int(cal_mask.sum()),
}
joblib.dump(bundle, STUDENT / "student_model.joblib")
print(f"Saved {STUDENT / 'student_model.joblib'}")

## Compare to your teammates

Copy your headline number into the team comparison table.
Then change ``CONFIG['model_family']`` and re-run to see
how the choice moves the needle.

Reference numbers from the rigorous pipeline (Phase 4 OOF
on the same 92-feature matrix; expect ±0.02 of these):

| model_family | roi_gt_2 OOF AUC |
|---|---|
| logistic | ~0.58 |
| random_forest | ~0.61 |
| svm_rbf | ~0.65 |
| xgboost | ~0.65 |

On the cal set (held out from training) the values land
slightly different from the OOF-CV value above — that's
normal and expected.